# 00 - 环境搭建

本 Notebook 指导你搭建 LangChain 开发环境，包括 Python 环境、依赖安装、API 配置和第一个程序。

## 学习目标

- ✅ 理解 LangChain 开发环境要求
- ✅ 安装 Python 和依赖管理工具 UV
- ✅ 配置 DeepSeek API Key
- ✅ 运行第一个 LangChain 程序
- ✅ 验证环境配置正确

---

## 1. 环境要求

### 1.1 Python 版本

**LangChain 要求：** Python >= 3.12

**为什么需要较新版本？**
- LangChain 1.0 使用了现代 Python 特性（类型注解、异步等）
- 部分依赖库（如 pydantic v2）需要 Python 3.12+

**检查 Python 版本：**

In [ ]:
!python --version

### 1.2 UV 包管理器

**术语：UV** — 一个现代 Python 包管理器，比 pip 快 10-100 倍，自动管理虚拟环境。

**安装 UV：**
```bash
# macOS/Linux
curl -LsSf https://astral.sh/uv/install.sh | sh

# Windows (PowerShell)
powershell -c "irm https://astral.sh/uv/install.ps1 | iex"
```

---

## 2. 项目依赖

### 2.1 核心依赖说明

| 依赖包 | 版本 | 用途 |
|--------|------|------|
| `langchain` | >=1.0 | LangChain 核心库 |
| `langchain-openai` | >=0.3 | OpenAI 兼容接口（支持 DeepSeek）|
| `langgraph` | >=1.0 | LangGraph Agent 框架 |
| `chromadb` | >=1.0 | 本地向量数据库（RAG 应用）|
| `pypdf` | >=5.0 | PDF 文档解析 |
| `tiktoken` | >=0.9 | Token 计数工具 |

**术语：依赖包（Dependency）** — 项目需要的外部库，通过包管理器安装。

### 2.2 安装依赖

**在项目根目录执行：**

```bash
# 1. 克隆项目后进入目录
cd langchain-learning

# 2. 安装依赖（UV 自动创建虚拟环境）
uv sync

# 3. 安装可编辑包（Notebook 导入需要）
uv pip install -e .
```

---

## 3. API 配置

### 3.1 DeepSeek API Key

**术语：API Key** — 访问大模型服务的密钥，类似"门禁卡"，证明你有权限调用 API。

**获取 DeepSeek API Key：**

1. 访问 DeepSeek 开放平台：https://platform.deepseek.com/
2. 注册/登录账号
3. 进入「API Keys」页面：https://platform.deepseek.com/api_keys
4. 点击「创建 API Key」
5. 复制生成的 Key（格式：`sk-xxxxx`）

**配置到项目：**

```bash
# 1. 复制示例配置文件
cp .env.example .env

# 2. 编辑 .env 文件，替换 API Key
# DEEPSEEK_API_KEY=sk-你的真实密钥
```

### 3.2 可选：LangSmith 追踪

**术语：LangSmith** — LangChain 官方的链路追踪平台，可视化调试 LLM 应用。

**启用追踪的好处：**
- 可视化 Chain/Agent 执行流程
- 查看 Prompt/Response 完整记录
- 分析 Token 消耗和耗时
- 调试复杂 Agent 行为

**配置（可选）：**

```bash
# 在 .env 中取消注释并填入
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=lsv2_xxx  # 从 https://smith.langchain.com 获取
LANGSMITH_PROJECT="langchain-learning"
```

---

## 4. 第一个 LangChain 程序

### 4.1 理解基本组件

**术语：LLM（Large Language Model）** — 大语言模型，如 GPT、Claude、DeepSeek。

**术语：Prompt Template** — 提示词模板，支持变量占位符（如 `{topic}`）。

**术语：Chain（链）** — 用 LCEL（LangChain Expression Language）语法 `|` 连接组件。

**工作流程：**

```mermaid
flowchart LR
    A[输入数据] --> B[Prompt 模板
填充变量]
    B --> C[LLM 调用]
    C --> D[返回结果]
    
    style A fill:#e1f5ff
    style B fill:#fff3e0
    style C fill:#f3e5f5
    style D fill:#e8f5e9
```

In [1]:
# 导入核心组件
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import os
from dotenv import load_dotenv

# 加载 .env 配置
load_dotenv()

# 初始化 LLM（使用 DeepSeek V3）
# 术语：model 参数指定模型名称
# 术语：base_url 指定 API 端点（DeepSeek 兼容 OpenAI 接口）
# 术语：api_key 从环境变量读取（安全实践）
llm = ChatOpenAI(
    model="deepseek-chat",  # deepseek-chat = V3, deepseek-reasoner = R1
    base_url="https://api.deepseek.com",
    api_key=os.getenv("DEEPSEEK_API_KEY")
)

# 定义 Prompt 模板
# 术语：{topic} 是占位符，调用时填充
prompt = ChatPromptTemplate.from_template(
    "请用一句话解释：{topic}"
)

# 用 LCEL 语法创建 Chain
# 术语：| 符号表示"管道"，数据从左流向右
chain = prompt | llm

### 4.2 调用 Chain

In [2]:
# 调用 Chain 并传入输入
# 术语：invoke() 触发 Chain 执行
# 术语：输入必须是字典，键名匹配 Prompt 占位符
result = chain.invoke({"topic": "Transformer"})

# 输出结果
print("AI 回答：")
print(result.content)

AI 回答：
Transformer是一种基于自注意力机制的神经网络架构，通过并行处理序列中所有位置的依赖关系来捕捉全局上下文，从而替代了传统的循环或卷积结构。


**术语：AIMessage** — LLM 返回的消息对象，`.content` 属性存储文本内容。

**执行流程：**

```mermaid
sequenceDiagram
    participant U as 用户代码
    participant P as Prompt Template
    participant L as LLM (DeepSeek)
    
    U->>P: invoke({"topic": "Transformer"})
    P->>P: 填充模板 → "请解释：Transformer"
    P->>L: 发送 Prompt
    L-->>P: 返回 AIMessage
    P-->>U: result.content
```

---

## 5. 验证环境

### 5.1 检查依赖版本

In [ ]:
import langchain
import langgraph
import chromadb

print(f"✅ LangChain: {langchain.__version__}")
print(f"✅ LangChain OpenAI: 已安装")
print(f"✅ LangGraph: 已安装")
print(f"✅ ChromaDB: 已安装")

### 5.2 测试 API 连接

In [ ]:
# 简单测试：调用 DeepSeek API
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(
    model="deepseek-chat",
    base_url="https://api.deepseek.com",
    api_key=os.getenv("DEEPSEEK_API_KEY")
)

# 发送简单消息
response = llm.invoke("说：API 连接成功！")
print(response.content)

### 5.3 测试项目 LLM 封装

**术语：封装（Wrapper）** — 隐藏复杂细节，提供简洁接口。项目提供 `get_llm()` 统一获取 LLM 实例。

In [ ]:
# 使用项目封装的 get_llm()
from langchain_learning.llm import get_llm

# 获取默认 LLM（DeepSeek V3）
llm = get_llm()

# 测试调用
response = llm.invoke("说：项目封装工作正常！")
print(response.content)

---

## 6. Notebook 使用

### 6.1 选择 Kernel

**术语：Kernel（内核）** — Jupyter 执行代码的运行环境，绑定了 Python 解释器和依赖。

**操作步骤：**
1. 打开任意 `.ipynb` 文件
2. 点击 VS Code 右上角的「Select Kernel」
3. 选择「Python Environments」→ 找到 `.venv` (UV 创建的虚拟环境)

**识别正确 Kernel：** 路径包含 `langchain-learning/.venv/bin/python`

### 6.2 常见问题

**Q1: `ModuleNotFoundError: No module named 'langchain'`**

**原因：** 选择了错误的 Kernel（没有安装依赖的系统 Python）

**解决：** 重新选择 `.venv` Kernel

---

**Q2: `AuthenticationError: Incorrect API key`**

**原因：** `.env` 文件中 API Key 配置错误

**解决：** 
1. 检查 `.env` 文件是否存在
2. 确认 `DEEPSEEK_API_KEY` 值格式正确（`sk-` 开头）
3. 重新加载 Kernel：`Kernel` → `Restart Kernel`

---

**Q3: UV 安装依赖失败**

**原因：** 网络问题或 UV 未正确安装

**解决：** 
```bash
# 手动安装依赖
uv pip install langchain langchain-openai langgraph chromadb
```

---

## 7. 环境架构图

```mermaid
flowchart TB
    subgraph 开发环境
        A[Python 3.12+] --> B[UV 包管理器]
        B --> C[.venv 虚拟环境]
    end
    
    subgraph 项目依赖
        D[langchain] --> E[langchain-openai]
        E --> F[langgraph]
        F --> G[chromadb]
        G --> H[其他依赖]
    end
    
    subgraph 配置文件
        I[.env<br/>API Key] --> J[pyproject.toml<br/>依赖声明]
    end
    
    C --> D
    C --> I
    
    K[Jupyter Notebook] --> C
    
    style A fill:#e3f2fd
    style B fill:#fff3e0
    style C fill:#f1f8e9
    style I fill:#fce4ec
    style K fill:#f3e5f5
```

---

## 8. 下一步

环境配置完成后，建议按以下顺序学习：

1. **01-core-concepts/00-langchain-overview** — LangChain 架构概览
2. **01-core-concepts/01-chat-models** — 聊天模型详解
3. **01-core-concepts/02-prompt-parsers** — 提示词工程
4. **01-core-concepts/03-lcel** — LangChain 表达式语言（LCEL）
5. **01-core-concepts/04-10** — 工具、流式输出、中间件、Agent、消息、回调、文档

**学习路径图：**

```mermaid
flowchart LR
    A[00-setup<br/>环境搭建] --> B[01-core-concepts<br/>核心概念]
    B --> C[02-rag<br/>RAG 应用]
    C --> D[03-engineering<br/>工程化]
    
    style A fill:#e1f5ff
    style B fill:#fff3e0
    style C fill:#f3e5f5
    style D fill:#e8f5e9
```

---

## 总结

本 Notebook 完成了：

- ✅ **Python 环境**：3.12+ + UV 包管理
- ✅ **依赖安装**：`uv sync` 一键安装所有依赖
- ✅ **API 配置**：DeepSeek API Key 配置到 `.env`
- ✅ **第一个程序**：使用 LCEL 语法 `prompt | llm` 构建简单 Chain
- ✅ **环境验证**：确认依赖版本和 API 连接正常

**核心概念回顾：**

| 概念 | 说明 |
|------|------|
| LLM | 大语言模型（如 DeepSeek V3）|
| Prompt Template | 带占位符的提示词模板 |
| Chain | 用 `|` 连接组件的工作流 |
| LCEL | LangChain Expression Language，`|` 语法 |
| invoke() | 触发 Chain 执行的方法 |

**下一步：** 继续学习 `01-core-concepts/00-langchain-overview`，了解 LangChain 架构！